In [1]:

import random

# Examination subjects and registered students
subjects = {
    "Maths": ["S1", "S2", "S3", "S4"],
    "Physics": ["S1", "S2", "S5"],
    "Chemistry": ["S3", "S4", "S6"],
    "Computer": ["S1", "S5", "S6"],
    "English": ["S2", "S4", "S5"]
}

# Available examination slots
slots = ["Slot 1", "Slot 2", "Slot 3", "Slot 4"]

# Examination halls and their capacities
rooms = {
    "Hall A": 5,
    "Hall B": 3
}

# ACO parameters
num_ants = 20
iterations = 50
alpha = 1
beta = 2
rho = 0.5
Q = 100

# Initialize pheromone
pheromone = {}

for subject in subjects:
    for slot in slots:
        for room in rooms:
            pheromone[(subject, slot, room)] = 1.0


# Check scheduling constraints
def is_valid(subject, slot, room, timetable):

    # Check room capacity
    if len(subjects[subject]) > rooms[room]:
        return False

    for other in timetable:

        other_slot, other_room = timetable[other]

        if slot == other_slot:

            # Check hall availability
            if room == other_room:
                return False

            # Check student conflicts
            if set(subjects[subject]) & set(subjects[other]):
                return False

    return True


# Calculate heuristic value
def heuristic(subject, room):

    unused = rooms[room] - len(subjects[subject])

    return 1 / (1 + unused)


# Initialize best solution
best_timetable = None
best_cost = float("inf")


# Main ACO algorithm
for iteration in range(iterations):

    iteration_best = None
    iteration_cost = float("inf")

    for ant in range(num_ants):

        timetable = {}

        # Schedule subjects with more students first
        order = sorted(
            subjects,
            key=lambda x: len(subjects[x]),
            reverse=True
        )

        for subject in order:

            candidates = []

            # Find valid assignments
            for slot in slots:
                for room in rooms:

                    if is_valid(subject, slot, room, timetable):

                        key = (subject, slot, room)

                        tau = pheromone[key]
                        eta = heuristic(subject, room)

                        probability = (
                            (tau ** alpha) * (eta ** beta)
                        )

                        candidates.append((key, probability))

            # No valid assignment
            if not candidates:
                break

            # Probability-based selection
            total = sum(p for key, p in candidates)

            choices = [key for key, p in candidates]
            weights = [p / total for key, p in candidates]

            selected = random.choices(
                choices, weights=weights, k=1
            )[0]

            _, slot, room = selected

            timetable[subject] = (slot, room)

        # Check complete timetable
        if len(timetable) != len(subjects):
            continue

        # Calculate scheduling penalty
        cost = 0

        for subject, (slot, room) in timetable.items():
            cost += rooms[room] - len(subjects[subject])

        if cost < iteration_cost:
            iteration_cost = cost
            iteration_best = timetable.copy()

    # Update global best solution
    if iteration_best is not None:

        if iteration_cost < best_cost:
            best_cost = iteration_cost
            best_timetable = iteration_best.copy()

    # Pheromone evaporation
    for key in pheromone:
        pheromone[key] *= (1 - rho)

    # Pheromone deposit
    if iteration_best is not None:

        deposit = Q / (1 + iteration_cost)

        for subject, (slot, room) in iteration_best.items():

            key = (subject, slot, room)

            pheromone[key] += deposit


# Display final timetable
if best_timetable:

    print("\nOPTIMIZED EXAMINATION TIMETABLE")
    print("-------------------------------------")

    print("Subject\t\tSlot\t\tHall")

    for subject, (slot, room) in best_timetable.items():

        print(subject, "\t\t", slot, "\t", room)

    print("-------------------------------------")
    print("Total Scheduling Penalty:", best_cost)

else:
    print("No feasible timetable found.")



OPTIMIZED EXAMINATION TIMETABLE
-------------------------------------
Subject		Slot		Hall
Maths 		 Slot 1 	 Hall A
Physics 		 Slot 2 	 Hall A
Chemistry 		 Slot 2 	 Hall B
Computer 		 Slot 4 	 Hall B
English 		 Slot 3 	 Hall B
-------------------------------------
Total Scheduling Penalty: 3
